# Pl@ntNet-300K — топ-100 видов, ResNet-50, экспорт в ONNX

Второй датасет для того же проекта: вместо студийных снимков Flowers-102 —
фотографии из приложения Pl@ntNet, снятые в поле телефонами. На выходе те же
два файла для папки `model/`: `model.onnx` и `labels.json`.

### Почему топ-100 видов, а не все 1 081

Полный датасет — 306 146 изображений с длинным хвостом: несколько видов
занимают большую часть снимков, а сотни видов представлены десятком фото. На
полном наборе ResNet-50 даёт порядка 80–90% top-1, и это потолок самого
датасета, а не вопрос времени обучения. Берём сто самых представленных видов
из доступного среза и ограничиваем число снимков на вид сверху — классы
выравниваются, а обучение влезает в одну сессию Colab.

### Почему зеркало Hugging Face, а не Zenodo

Оригинал на Zenodo — один zip на 31.7 ГБ, и отдаётся он медленно: замеры из
Colab дают 0.7 МБ/с против 11 МБ/с у Hugging Face, то есть часы против минут.

Но дело не только в скорости. На Hugging Face датасет разложен на 51 файл
parquet, **упорядоченных по видам**: каждый файл содержит непрерывный кусок
видов целиком. Значит качать всё не нужно — первых тринадцати файлов (≈6.5 ГБ)
хватает примерно на 270 видов со всеми их снимками, а нам из них нужны сто.

### Сколько это займёт

| Этап | Время |
|---|---|
| Скачать 13 файлов parquet (≈6.5 ГБ) | ~10 мин |
| Собрать подмножество с пережатием | ~5 мин |
| Обучение, 6 + 12 эпох | ~30 мин |

Подмножество сохраняется на Google Drive (1–2 ГБ). При повторном запуске
ноутбук находит его и пропускает скачивание.

### Перед запуском

`Runtime` → `Change runtime type` → **GPU**. Затем `Run all`.

## 0. Окружение

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU НЕ НАЙДЕН: Runtime -> Change runtime type -> GPU"
!pip install -q onnx onnxruntime

In [ ]:
import io, json, os, shutil, tarfile, time, random
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torchvision
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet50, ResNet50_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")
if DEVICE == "cpu":
    print("ВНИМАНИЕ: на CPU обучение займёт много часов. Включи GPU.")

## 1. Настройки

`MAX_TRAIN_PER_SPECIES` — главный рычаг. Он же лечит перекос классов: без
ограничения самый частый вид дал бы в сотни раз больше примеров, чем сотый.

In [ ]:
NUM_SPECIES = 100              # сколько видов берём (по числу снимков)
MAX_TRAIN_PER_SPECIES = 500    # потолок снимков на вид в обучении
MAX_EVAL_PER_SPECIES = 100     # потолок на вид в val/test — чтобы оценка шла быстро
TRAIN_SHARDS = 13              # сколько файлов parquet качаем (из 51)

IMG_SIZE = 224
RESIZE_SIZE = 256
STORED_SIZE = 256              # до какого размера пережимаем при распаковке

HF_REPO = "mikehemberger/plantnet300K"
# Научные названия видов: в репозитории Pl@ntNet этого файла больше нет,
# но он есть на зеркале рядом с данными.
SPECIES_META = "data/train/metadata.csv"

WORK = Path("/content")
SHARD_DIR = WORK / "parquet"
SUBSET_DIR = WORK / "plantnet_subset"
CACHE_NAME = f"plantnet_top{NUM_SPECIES}_{MAX_TRAIN_PER_SPECIES}.tar"

## 2. Google Drive как кеш

Не обязателен, но рекомендуется: на Диск кладётся только собранное
подмножество (1–2 ГБ), а не скачанные файлы parquet. Если Диск не подключать,
при обрыве сессии данные придётся качать заново.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_PATH = Path("/content/drive/MyDrive") / CACHE_NAME
    print(f"кеш подмножества: {CACHE_PATH}")
except Exception as exc:
    CACHE_PATH = WORK / CACHE_NAME
    print(f"Диск недоступен ({exc}), кеш только на локальном диске сессии")

HAVE_CACHE = CACHE_PATH.exists()
print("кеш найден, скачивание не потребуется" if HAVE_CACHE
      else "кеша нет — качаем файлы parquet с Hugging Face")

## 3. Скачивание файлов parquet

Качаем только первые `TRAIN_SHARDS` файлов обучающей выборки. Файлы val и
test берём по одному, пока не покроем диапазон меток, попавший в обучение:
сплиты упорядочены по видам одинаково, поэтому лишнего не скачается.

`hf_hub_download` сам докачивает прерванное и кеширует — повторный запуск
ячейки после обрыва продолжит с места остановки.

In [ ]:
# Имена файлов в зеркале не совпадают с именами сплитов: валидация лежит
# как validation-*, а не val-*. Поэтому разбираем оглавление репозитория,
# а не угадываем префиксы.
SPLIT_PREFIX = {"train": "train", "val": "validation", "test": "test"}

if not HAVE_CACHE:
    available = HfApi().list_repo_files(HF_REPO, repo_type="dataset")
    SHARDS = {}
    for split, prefix in SPLIT_PREFIX.items():
        SHARDS[split] = sorted(
            f for f in available
            if f.startswith(f"data/{prefix}-") and f.endswith(".parquet")
        )
        assert SHARDS[split], (
            f"в репозитории нет файлов для сплита {split} (искали data/{prefix}-*.parquet). "
            f"Похожие пути: {[f for f in available if prefix in f][:3]}"
        )
        print(f"{split}: {len(SHARDS[split])} файлов")

    def fetch(path):
        return hf_hub_download(HF_REPO, path, repo_type="dataset",
                               local_dir=str(SHARD_DIR))

    started = time.time()
    with ThreadPoolExecutor(max_workers=4) as pool:
        train_files = list(pool.map(fetch, SHARDS["train"][:TRAIN_SHARDS]))
    volume = sum(Path(f).stat().st_size for f in train_files)
    print(f"\nскачано {volume / 1e9:.1f} ГБ за {(time.time() - started) / 60:.1f} мин")

## 4. Выбор видов

Имена классов лежат прямо в метаданных parquet — это ID видов Pl@ntNet,
1 081 штука в том же порядке, что и числовые метки.

Последний вид в последнем скачанном файле отбрасываем: он почти наверняка
разрезан границей файлов, и его снимки посчитались бы неполностью.

In [ ]:
if not HAVE_CACHE:
    schema_meta = pq.ParquetFile(train_files[0]).schema_arrow.metadata
    hf_info = json.loads(schema_meta[b"huggingface"].decode())
    SPECIES_IDS = hf_info["info"]["features"]["label"]["names"]
    print(f"видов в датасете: {len(SPECIES_IDS)}")

    counts = Counter()
    for path in train_files:
        counts.update(pq.read_table(path, columns=["label"]).column("label").to_pylist())

    truncated = max(counts)          # вид, разрезанный границей файлов
    del counts[truncated]
    print(f"видов целиком в скачанном срезе: {len(counts)}")

    CHOSEN = [label for label, _ in counts.most_common(NUM_SPECIES)]
    CHOSEN_SET = set(CHOSEN)
    print(f"отобрано: {len(CHOSEN)} видов")
    print(f"снимков у самого частого: {counts[CHOSEN[0]]:,}")
    print(f"снимков у сотого:         {counts[CHOSEN[-1]]:,}")
    kept = sum(min(counts[l], MAX_TRAIN_PER_SPECIES) for l in CHOSEN)
    print(f"после ограничения в {MAX_TRAIN_PER_SPECIES} на вид останется {kept:,} снимков")

## 5. Сборка подмножества

Разворачиваем выбранные виды в обычную структуру папок, какую понимает
`ImageFolder`, попутно пережимая до короткой стороны 256 px. Узкое место
обучения в Colab — декодирование JPEG на двух ядрах, а не GPU, поэтому
уменьшение картинок один раз окупается на каждой эпохе.

In [ ]:
def shrink(raw):
    image = Image.open(io.BytesIO(raw)).convert("RGB")
    width, height = image.size
    scale = STORED_SIZE / min(width, height)
    if scale < 1:
        image = image.resize((max(1, round(width * scale)), max(1, round(height * scale))),
                             Image.BILINEAR)
    return image


def write_split(paths, split, limit):
    kept = Counter()
    pool = ThreadPoolExecutor(max_workers=os.cpu_count() * 2)
    futures = []

    def save(raw, dest):
        shrink(raw).save(dest, "JPEG", quality=90)

    for path in paths:
        parquet = pq.ParquetFile(path)
        for batch in parquet.iter_batches(batch_size=256, columns=["image", "label"]):
            labels = batch.column("label").to_pylist()
            images = batch.column("image").to_pylist()
            for image, label in zip(images, labels):
                if label not in CHOSEN_SET or kept[label] >= limit:
                    continue
                out_dir = SUBSET_DIR / split / SPECIES_IDS[label]
                out_dir.mkdir(parents=True, exist_ok=True)
                futures.append(pool.submit(save, image["bytes"], out_dir / f"{kept[label]:04d}.jpg"))
                kept[label] += 1

    pool.shutdown(wait=True)
    failed = sum(1 for f in futures if f.exception())
    print(f"{split}: {sum(kept.values()):,} снимков, {len(kept)} видов"
          + (f", с ошибкой {failed}" if failed else ""))

    # Пустой сплит — это ошибка в фильтрах, а не допустимый случай.
    # Падаем здесь, пока скачанные файлы ещё не удалены.
    assert kept, f"в сплит {split} не попало ни одного снимка"
    return sum(kept.values())


def collect_eval(split):
    """Качаем файлы сплита по одному, пока не покроем нужные метки."""
    paths, ceiling = [], max(CHOSEN)
    for remote in SHARDS[split]:
        local = fetch(remote)
        paths.append(local)
        seen = pq.read_table(local, columns=["label"]).column("label").to_pylist()
        if max(seen) > ceiling:
            break
    return paths


if not HAVE_CACHE:
    started = time.time()
    write_split(train_files, "train", MAX_TRAIN_PER_SPECIES)
    for split in ("val", "test"):
        write_split(collect_eval(split), split, MAX_EVAL_PER_SPECIES)
    print(f"подмножество собрано за {(time.time() - started) / 60:.1f} мин")

    with tarfile.open(CACHE_PATH, "w") as tar:
        tar.add(SUBSET_DIR, arcname=SUBSET_DIR.name)
    print(f"кеш сохранён: {CACHE_PATH.stat().st_size / 1e9:.2f} ГБ")

    shutil.rmtree(SHARD_DIR, ignore_errors=True)   # parquet больше не нужны
else:
    with tarfile.open(CACHE_PATH) as tar:
        tar.extractall(WORK)
    print("подмножество восстановлено из кеша")

for split in ("train", "val", "test"):
    folders = sorted((SUBSET_DIR / split).iterdir())
    images = list((SUBSET_DIR / split).rglob("*.jpg"))
    print(f"{split}: {len(images):,} снимков, {len(folders)} видов")

## 6. Датасет

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_set = ImageFolder(SUBSET_DIR / "train", transform=train_tf)
val_set = ImageFolder(SUBSET_DIR / "val", transform=eval_tf)
test_set = ImageFolder(SUBSET_DIR / "test", transform=eval_tf)

assert train_set.classes == val_set.classes == test_set.classes, "сплиты разошлись по классам"
NUM_CLASSES = len(train_set.classes)
print(f"классов: {NUM_CLASSES} | train {len(train_set):,} | val {len(val_set):,} | test {len(test_set):,}")

BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

В папках лежат числовые ID видов — в чате такое показывать нельзя. Научные
названия берём из `metadata.csv` рядом с данными: файл весит 66 МБ и
покрывает все 1 081 вид.

In [ ]:
import csv

meta_path = hf_hub_download(HF_REPO, SPECIES_META, repo_type="dataset",
                            local_dir=str(WORK / "meta"))

species_id_2_name = {}
with open(meta_path, newline="") as handle:
    for row in csv.DictReader(handle):
        species_id_2_name.setdefault(row["class"], row["species"])

# ImageFolder сортирует папки, и порядок classes задаёт индексы меток —
# имена обязаны идти ровно в этом же порядке.
CLASS_NAMES = [species_id_2_name.get(sp, f"species {sp}") for sp in train_set.classes]
missing = sum(1 for n in CLASS_NAMES if n.startswith("species "))
print(f"названий получено: {len(CLASS_NAMES) - missing} из {len(CLASS_NAMES)}")
print("первые пять:", CLASS_NAMES[:5])

In [ ]:
import matplotlib.pyplot as plt

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax in axes.flat:
    img, label = train_set[random.randrange(len(train_set))]
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[label], fontsize=7)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Модель и обучение

In [ ]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

def set_backbone_trainable(model, names):
    for param in model.parameters():
        param.requires_grad = False
    for name in list(names) + ["fc"]:
        for param in getattr(model, name).parameters():
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"обучаемых параметров: {trainable:,} из {total:,} ({100 * trainable / total:.1f}%)")

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


@torch.no_grad()
def evaluate(model, loader):
    """Возвращает top-1 и macro-average top-1 — среднее по видам.

    На длиннохвостых данных эти две цифры расходятся: обычная точность
    выезжает на частых видах, macro показывает, что происходит с редкими.
    """
    model.eval()
    correct = torch.zeros(NUM_CLASSES)
    total = torch.zeros(NUM_CLASSES)
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        preds = model(images).argmax(dim=1).cpu()
        for label, pred in zip(labels, preds):
            total[label] += 1
            correct[label] += int(label == pred)
    seen = total > 0
    return (correct.sum() / total.sum()).item(), (correct[seen] / total[seen]).mean().item()


def train_phase(model, epochs, lr, trainable_blocks, label):
    set_backbone_trainable(model, trainable_blocks)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader))
    scaler = torch.amp.GradScaler(DEVICE, enabled=(DEVICE == "cuda"))

    for epoch in range(1, epochs + 1):
        model.train()
        started = time.time()
        running_loss = seen = 0
        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=(DEVICE == "cuda")):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item() * labels.numel()
            seen += labels.numel()
        print(f"[{label}] эпоха {epoch}/{epochs}  loss {running_loss / seen:.4f}"
              f"  {time.time() - started:.0f} c")
    return model

### Фаза 1 — только классификатор

In [ ]:
train_phase(model, epochs=6, lr=1e-3, trainable_blocks=[], label="фаза 1")
acc1, macro1 = evaluate(model, val_loader)
print(f"\nval после фазы 1: top-1 {acc1:.4f} | macro {macro1:.4f}")

### Фаза 2 — разморозка layer3 и layer4

In [ ]:
train_phase(model, epochs=12, lr=1e-4, trainable_blocks=["layer3", "layer4"], label="фаза 2")
acc2, macro2 = evaluate(model, val_loader)
print(f"\nval после фазы 1: top-1 {acc1:.4f} | macro {macro1:.4f}")
print(f"val после фазы 2: top-1 {acc2:.4f} | macro {macro2:.4f}")

In [ ]:
acc_test, macro_test = evaluate(model, test_loader)
print(f"ТЕСТ: top-1 {acc_test:.4f} | macro-average {macro_test:.4f}")

torch.save(model.state_dict(), "resnet50_plantnet100.pt")
print("веса сохранены: resnet50_plantnet100.pt")

## 8. Экспорт в ONNX

Ровно тот же контракт, что и у модели на Flowers-102: нормализация и softmax
внутри графа, вход `1x3x224x224` в `[0,1]`. Сервер подхватит новую модель без
единой правки в коде.

In [ ]:
class ExportWrapper(nn.Module):
    """Принимает RGB в [0,1], внутри нормализует и отдаёт вероятности."""

    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, pixel_values):
        return torch.softmax(self.model((pixel_values - self.mean) / self.std), dim=1)


wrapper = ExportWrapper(model, IMAGENET_MEAN, IMAGENET_STD).to("cpu").eval()
dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)

ONNX_PATH = "plantnet100.onnx"
export_kwargs = dict(
    input_names=["pixel_values"],
    output_names=["probabilities"],
    opset_version=17,
    dynamic_axes={"pixel_values": {0: "batch"}, "probabilities": {0: "batch"}},
)
try:
    torch.onnx.export(wrapper, dummy, ONNX_PATH, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(wrapper, dummy, ONNX_PATH, **export_kwargs)

print(f"{ONNX_PATH}: {Path(ONNX_PATH).stat().st_size / 1e6:.1f} МБ")

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

# Берём случайные снимки из теста, а не первый батч: он отсортирован по классам.
indices = random.sample(range(len(test_set)), 8)
images = torch.stack([test_set[i][0] for i in indices])
labels = [test_set[i][1] for i in indices]
batch = (images * std + mean).clamp(0, 1)

with torch.no_grad():
    torch_probs = wrapper(batch).numpy()
onnx_probs = session.run(None, {"pixel_values": batch.numpy()})[0]

max_diff = np.abs(torch_probs - onnx_probs).max()
print(f"максимальное расхождение вероятностей: {max_diff:.2e}")
assert max_diff < 1e-4, "torch и onnxruntime расходятся — экспорт сломан"

print("\nпредсказания onnxruntime:")
for i, true_label in enumerate(labels):
    top = int(onnx_probs[i].argmax())
    mark = "+" if top == true_label else "-"
    print(f"  {mark} {CLASS_NAMES[top][:34]:<34} {onnx_probs[i][top]:.3f}"
          f"   (истина: {CLASS_NAMES[true_label][:30]})")

## 9. Метаданные и выгрузка

In [ ]:
meta = {
    "architecture": "resnet50",
    "dataset": f"PlantNet-{NUM_SPECIES}",
    "num_classes": NUM_CLASSES,
    "classes": CLASS_NAMES,
    "accuracy": {
        "test_top1_head_only": round(acc1, 4),
        "test_top1_finetuned": round(acc_test, 4),
        "test_macro_top1": round(macro_test, 4),
        "test_size": len(test_set),
    },
    "preprocessing": {
        "resize_shorter_side": RESIZE_SIZE,
        "center_crop": IMG_SIZE,
        "scale": "divide by 255, range [0,1]",
        "normalization": "inside the ONNX graph",
        "input_name": "pixel_values",
        "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
        "output_name": "probabilities",
        "output": "softmax probabilities",
    },
}

Path("plantnet100.labels.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in meta.items() if k != "classes"}, indent=2, ensure_ascii=False))

In [ ]:
from google.colab import files

files.download("plantnet100.onnx")
files.download("plantnet100.labels.json")

Оба файла — в папку `model/` проекта, рядом с теми же именами. Сервер читает
число классов из графа, а названия и метрики из `labels.json`, поэтому
переключение между моделями Flowers-102 и Pl@ntNet сводится к замене двух
файлов и перезапуску `uvicorn`.

Порог «не уверен» в `app/inference.py` стоит проверить заново: он подбирался
под 102 класса Flowers-102.